In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1.   PROJECT_ROOT를 실행환경에 맞게 설정하고 압축 더미 파일들 생성해서 밑에 train.py 실행하시면 10분안에 6에폭 도는 코드 확인 가능합니다!

2.   데이터 경로는 config을 통해 변경가능하고 압축 파일 인식, 압축 해제 시 폴더명, 데이터 로드시 인식 폴더명 등 data 키에서 병경 가능합니다!



In [3]:
!pip -q install "albumentations>=1.3.1" opencv-python-headless scikit-learn pyyaml transformers peft

더미 압축 파일 생성(tar 도 포함)

In [5]:
# ============================================
# Dummy data builder for NEW simple pipeline
# - creates 4 zips:
#   train_images.zip, val_images.zip, train_videos.zip, val_videos.zip
# - each zip contains: Real/..., Fake/...
# - videos zips also contain images (frames already extracted)
# ============================================

import os, shutil, zipfile
from pathlib import Path
import numpy as np
import cv2

# ------------------------------------------------
# 1) project root (너 환경에 맞게 수정)
# ------------------------------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/Dacon_Hecto/5th_submission")  # <= 수정 가능
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)
print("CWD:", Path.cwd())

# ------------------------------------------------
# 2) target paths
# ------------------------------------------------
TRAIN_DATA = Path("./train_data")
ZIPS_DIR = TRAIN_DATA / "train_data_zips"
ZIPS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------
# 3) helper utils
# ------------------------------------------------
IMG_H, IMG_W = 512, 512  # 이후 448 resize니까 512면 충분

def wipe(p: Path):
    if p.exists():
        shutil.rmtree(p, ignore_errors=True)

def make_dummy_jpg(path: Path, seed: int):
    rng = np.random.default_rng(seed)
    img = rng.integers(0, 255, size=(IMG_H, IMG_W, 3), dtype=np.uint8)
    path.parent.mkdir(parents=True, exist_ok=True)
    ok = cv2.imwrite(str(path), img)
    if not ok:
        raise RuntimeError(f"cv2.imwrite failed: {path}")

def build_zip_from_dir(src_dir: Path, zip_path: Path):
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for f in src_dir.rglob("*"):
            if f.is_file():
                zf.write(f, arcname=str(f.relative_to(src_dir)))

# ------------------------------------------------
# 4) create 4 split folders, then zip them
#    IMPORTANT: extracted folder should be:
#      ./train_data/train_images/Real/*.jpg
# so zip content must be Real/... Fake/... at top level.
# ------------------------------------------------
TMP = Path("./_dummy_build_v2")
wipe(TMP)
TMP.mkdir(parents=True, exist_ok=True)

def make_split_folder(split_name: str, n_real: int, n_fake: int, seed0: int):
    base = TMP / split_name
    wipe(base)
    (base / "Real").mkdir(parents=True, exist_ok=True)
    (base / "Fake").mkdir(parents=True, exist_ok=True)

    # Real
    for i in range(n_real):
        make_dummy_jpg(base / "Real" / f"{split_name}_real_{i:05d}.jpg", seed=seed0 + i)

    # Fake
    for i in range(n_fake):
        make_dummy_jpg(base / "Fake" / f"{split_name}_fake_{i:05d}.jpg", seed=seed0 + 100000 + i)

    return base

# ✅ 데이터량: batch_size=48 drop_last=True 고려해서 넉넉히
# train_images / train_videos 합치면 train이 충분히 커지고,
# val도 auc 계산 가능하도록 충분히.
train_images_dir = make_split_folder("train_images", n_real=200, n_fake=200, seed0=10)
val_images_dir   = make_split_folder("val_images",   n_real=80,  n_fake=80,  seed0=1000000)

# videos도 "이미지"로만 구성 (frames)
train_videos_dir = make_split_folder("train_videos", n_real=200, n_fake=200, seed0=2000000)
val_videos_dir   = make_split_folder("val_videos",   n_real=80,  n_fake=80,  seed0=3000000)

# build zips
build_zip_from_dir(train_images_dir, ZIPS_DIR / "train_images.zip")
build_zip_from_dir(val_images_dir,   ZIPS_DIR / "val_images.zip")
build_zip_from_dir(train_videos_dir, ZIPS_DIR / "train_videos.zip")
build_zip_from_dir(val_videos_dir,   ZIPS_DIR / "val_videos.zip")

print("\n[Created dummy zips]")
for p in sorted(ZIPS_DIR.glob("*.zip")):
    print(" -", p.name, f"({p.stat().st_size:,} bytes)")

print("\nNOTE: These zips contain top-level Real/ and Fake/ folders.")
print("Expected extract result:")
print("  ./train_data/train_images/Real/*.jpg")
print("  ./train_data/train_images/Fake/*.jpg")
print("  ... similarly for val_images, train_videos, val_videos")


CWD: /content/drive/.shortcut-targets-by-id/1XmFV5eDfRBdOt2-3OwF7VRzBy3jH9cyp/5th_submission

[Created dummy zips]
 - kodf_video_frame_v2.zip (131,525,549 bytes)
 - organized_val.zip (4,557,757,739 bytes)
 - pexel_video_frame.zip (68,663,298 bytes)
 - refined_dataset.zip (34,441,897 bytes)
 - train.zip (558,110,931 bytes)
 - train_images.zip (122,766,901 bytes)
 - train_videos.zip (122,773,247 bytes)
 - val_images.zip (49,102,841 bytes)
 - val_videos.zip (49,108,413 bytes)
 - video_frames.zip (371,515,305 bytes)

NOTE: These zips contain top-level Real/ and Fake/ folders.
Expected extract result:
  ./train_data/train_images/Real/*.jpg
  ./train_data/train_images/Fake/*.jpg
  ... similarly for val_images, train_videos, val_videos


혹시 모를 루트 재선언

In [4]:
import os, random, zipfile, tarfile, shutil
from pathlib import Path
import numpy as np

# 네 프로젝트가 /content/your_submission 에 있다고 가정 (필요 시 수정)
PROJECT_ROOT = Path("/content/drive/MyDrive/Dacon_Hecto/5th_submission").resolve()  # <-- 여기만 네 상황에 맞게
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)

print("CWD:", Path.cwd())

CWD: /content/drive/.shortcut-targets-by-id/1XmFV5eDfRBdOt2-3OwF7VRzBy3jH9cyp/5th_submission


더미로 실행 해보는 실험 코드, 실제 데이터 넣어봐도 됨

In [ ]:
import os
os.environ["RUNTIME_ROOT"]="/content"
os.environ["HF_TOKEN"] = "YOUR_HUGGINGFACE_TOKEN"

!python train.py --config config/config.yaml


[RUNTIME] runtime_root: /content
[CFG] cfg_path: /content/drive/.shortcut-targets-by-id/1XmFV5eDfRBdOt2-3OwF7VRzBy3jH9cyp/5th_submission/config/config.yaml
[CFG] cfg_dir : /content/drive/.shortcut-targets-by-id/1XmFV5eDfRBdOt2-3OwF7VRzBy3jH9cyp/5th_submission/config
[CFG] cwd     : /content/drive/.shortcut-targets-by-id/1XmFV5eDfRBdOt2-3OwF7VRzBy3jH9cyp/5th_submission
[Extract] zip: train_full_data.zip -> /content/train_data/train_full_data
[Extract] done -> /content/train_data/train_full_data
[Check] extracted root exists: True
Loading weights: 100% 415/415 [00:23<00:00, 17.64it/s, Materializing param=norm.weight]
[Model] total=311,781,377 trainable=8,651,777
[Dataset Source Scan]

[REAL]
 - /content/train_data/train_full_data/Real-img/Real-img  ->  38,582
 - /content/train_data/train_full_data/Real  ->  30,000
 - /content/train_data/train_full_data/ffhq  ->  70,000
 - /content/train_data/train_full_data/video_frames/real  ->  4,361
 - /content/train_data/train_full_data/refined_datas